# **Optimización TP1**: Perfil de Desempeño

Diseñado por Dolan y Moré en 2001, es una herramienta que se emplea para comparar la aptitud de distintos algoritmos para resolver un conjunto de problemas.

Dados un conjunto de problemas $P$ y un conjunto de algoritmos $S$, definimos como $c_{s,p}$ el costo de resolver el problema $p \in P$ con el algoritmo $s \in S$. Si el algoritmo $s$ no puede resolver el problema $p$, se define $c_{s,p} = c_{max}$ con $c_{max} \in \mathbb{R}$ un valor adecuado de manera tal que:

$$c_{s,p} = c_{max} \Leftrightarrow \textbf{el algoritmo } s \textbf{ no resuelve el problema } p$$

Se asume que al menos un algoritmo resuelve el problema $p$. Se define la tasa de desempeño relativa como:

$$r_{s,p} = \frac{c_{s,p}}{\min\{c_{j,p}: j \in S\}}$$

Observar que $r_{s,p} \geq 1$ y, si $r_{s,p} = 1$, entonces el algoritmo $s$ es uno de los mejores (o el mejor) para resolver el problema $p$. Mientras mayor sea $r_{s,p}$, peor es el desempeño de $s$ al resolver $p$. Finalmente, se define la función de desempeño $\rho_s : [1, \infty) \to [0, 1]$ para el algoritmo $s$ como:

$$\rho_s(\tau) = \frac{\#\{p \in P : r_{s,p} \leq \tau\}}{\#P}$$

Así, $\rho_s(\tau)$ es el porcentaje de problemas que el algoritmo $s$ resuelve con hasta $\tau$ veces el costo del algoritmo más eficiente. Por ejemplo, $\rho_s(1)$ es la proporción de problemas que el algoritmo $s$ resuelve con el menor costo.

Si se define:

$$r_{max} = \max_{s \in S,\, p \in P\,:\, c_{s,p} < M} r_{s,p}$$

se tiene que $\rho_s(r_{max})$ es el número de problemas resueltos por el algoritmo $s$. El valor $\rho_s(1)$ se denomina la eficiencia de $s$ y $\rho_s(r_{max})$ es su robustez.

La definición del costo está relacionada a la medida de desempeño que se quiera estudiar. Se puede considerar como **costo** al **tiempo que necesita el algoritmo para resolver el problema, a la cantidad de iteraciones, a la cantidad de veces que es evaluada la función objetivo**, etc.

Para analizar el perfil de desempeño se suelen graficar en conjunto las funciones $\rho_s$ para cada $s \in S$.

## **Ejercicios**: Análisis de desempeño

In [45]:
using Pkg
Pkg.add(["DataFrames", "LinearAlgebra", "Plots"])

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`


In [1]:
include("benchmarks.jl")
using LinearAlgebra, Plots, DataFrames

### **Ejercicio 5**: Métodos de descenso

In [38]:
# Método de descenso por gradiente con paso constante
function gradiente_constante(x₀, f, ∇f, η=0.0001, max_iter=10000)
    # Condiciones iniciales
    xk = x₀; gk = ∇f(xk); dk = -gk
    iter = 0
    status = "MAX_ITER"

    # Iteraciones
    while iter < max_iter
        # Criterio de parada
        if norm(gk) < 1e-6
            status = "CONVERGIO"
            break

        end
        # Actualización
        xk = (xk + η .* dk); gk = ∇f(xk); dk = -gk
        iter += 1
        
    end
    return status, iter, norm(gk), xk
    
end

gradiente_constante (generic function with 3 methods)

In [39]:
function armijo(a₀, xk, dk, f, ∇f)
    # Busqueda lineal
    h(a) = f(xk + a .* dk)
    dh₀  = ∇f(xk)' * dk

    # Itero para buscar ak
    ak = a₀; iter = true
    while iter
        # Paso muy largo
        if (h(ak) > h(0) + dh₀*ak/2)
            ak = ak/2
        # Paso muy corto
        elseif (h(2*ak) <= h(0) + dh₀*ak) 
            ak = 10*ak
        # Cumple Armijo
        else                               
            return ak
            
        end
    end
end

armijo (generic function with 1 method)

In [40]:
# Método de descenso por gradiente con búsqueda lineal (Método de Armijo)
function gradiente_armijo(x₀, f, ∇f, max_iter=10000)
    # Condiciones iniciales
    xk = x₀; gk = ∇f(xk); ak = 1/2; dk = -gk
    iter = 0
    status = "MAX_ITER"

    # Iteraciones
    while iter < max_iter
        # Criterio de parada
        if norm(gk) < 1e-6
            status = "CONVERGIO"
            break

        end
        # Actualización
        ak = armijo(ak, xk, dk, f, ∇f)
        xk = (xk + ak .* dk); gk = ∇f(xk); dk = -gk
        iter += 1
        
    end
    return status, iter, norm(gk), xk
    
end

gradiente_armijo (generic function with 2 methods)

In [41]:
# Método de Newton
function newton(x₀, f, ∇f, Hf, max_iter=10000)
    # Condiciones iniciales
    xk = x₀; gk = ∇f(xk)
    iter = 0
    status = "MAX_ITER"

    # Iteraciones
    while iter < max_iter
        # Criterio de parada
        if norm(gk) < 1e-6
            status = "CONVERGIO"
            break
            
        end
        # Try descomposición de Cholesky
        C = cholesky(Hf(xk), check=false)

        # Actualización: Si Cholesky tuvo éxito
        if issuccess(C)      
            dk = C \ (-gk)   
            xk = xk + dk; gk = ∇f(xk)
            iter += 1 
        # Termino: Si Cholesky falló
        else 
            status = "NO_CONVERGIO"
            break
            
        end
    end
    return status, iter, norm(gk), xk
    
end

newton (generic function with 2 methods)

In [42]:
# Método de gradientes conjugados no-lineal (Método Fletcher–Reeves)
function gradientes_conjugados(x₀, f, ∇f, max_iter=10000)
    # Condiciones iniciales
    xk = x₀; gk = ∇f(xk); ak = 1/2; dk = -gk
    n = length(xk)
    iter = 0
    status = "MAX_ITER"

    # Iteraciones
    while iter < max_iter
        # Criterio de parada
        if norm(gk) < 1e-6
            status = "CONVERGIO"
            break
            
        end
        # Actualización
        ak = armijo(ak, xk, dk, f, ∇f);
        xk = xk + ak .* dk; βk = (norm(∇f(xk))/norm(gk))^2; gk = ∇f(xk)
        dk = (iter%n == n-1) ? -gk : -gk + βk .* dk
        iter += 1
         
    end
    return status, iter, norm(gk), xk
    
end

gradientes_conjugados (generic function with 2 methods)

### **Ejercicio 6**: Benchmarks

In [43]:
# Puntos iniciales
x0_2d = [1.0, 1.0]
x0_rb = [-1.2, 1.0]
x0_wd = [-3.0, -1.0, -3.0, -1.0]
x0_fr = [0.5, -2.0]

# Tabla de funciones
benchmarks = [
    ("Sphere",      sphere,      sphere_g,      sphere_h,      x0_2d),
    ("Rosenbrock",  rosenbrock,  rosenbrock_g,  rosenbrock_h,  x0_rb),
    ("Rastrigin",   rastrigin,   rastrigin_g,   rastrigin_h,   x0_2d),
    ("Wood",        wood,        wood_g,        wood_h,        x0_wd),
    ("Froth",       froth,       froth_g,       froth_h,       x0_fr),
    
]

# Tabla de métodos
metodos = [
    ("Grad. Constante",  (f, ∇f, Hf, x0) -> gradiente_constante(x0, f, ∇f)),
    ("Grad. Armijo",     (f, ∇f, Hf, x0) -> gradiente_armijo(x0, f, ∇f)),
    ("Newton",           (f, ∇f, Hf, x0) -> newton(x0, f, ∇f, Hf)),
    ("Grad. Conjugados", (f, ∇f, Hf, x0) -> gradientes_conjugados(x0, f, ∇f)),

]

# Tabla de resultados
resultados = DataFrame(
    Funcion  = String[],
    Metodo   = String[],
    Status   = String[],
    Iters    = Int[],
    NormGrad = Float64[],
    xFinal   = Vector{Float64}[]

)

Row,Funcion,Metodo,Status,Iters,NormGrad,xFinal
,String,String,String,Int64,Float64,Array…


In [44]:
# Evaluaciones
for (nombre, f, ∇f, Hf, x0) in benchmarks
    for (nombre_met, metodo) in metodos
        print("Ejecutando $nombre_met en $nombre... ")
        s, i, g, xk = metodo(f, ∇f, Hf, x0)
        push!(resultados, (nombre, nombre_met, s, i, g, xk))
        println("$s en $i iteraciones")
        
    end
end

Ejecutando Grad. Constante en Sphere... MAX_ITER en 10000 iteraciones
Ejecutando Grad. Armijo en Sphere... CONVERGIO en 1 iteraciones
Ejecutando Newton en Sphere... CONVERGIO en 1 iteraciones
Ejecutando Grad. Conjugados en Sphere... CONVERGIO en 1 iteraciones
Ejecutando Grad. Constante en Rosenbrock... MAX_ITER en 10000 iteraciones
Ejecutando Grad. Armijo en Rosenbrock... CONVERGIO en 960 iteraciones
Ejecutando Newton en Rosenbrock... CONVERGIO en 6 iteraciones
Ejecutando Grad. Conjugados en Rosenbrock... CONVERGIO en 217 iteraciones
Ejecutando Grad. Constante en Rastrigin... CONVERGIO en 368 iteraciones
Ejecutando Grad. Armijo en Rastrigin... CONVERGIO en 1 iteraciones
Ejecutando Newton en Rastrigin... CONVERGIO en 2 iteraciones
Ejecutando Grad. Conjugados en Rastrigin... CONVERGIO en 1 iteraciones
Ejecutando Grad. Constante en Wood... MAX_ITER en 10000 iteraciones
Ejecutando Grad. Armijo en Wood... CONVERGIO en 1187 iteraciones
Ejecutando Newton en Wood... NO_CONVERGIO en 7 iteracion

In [45]:
resultados

Row,Funcion,Metodo,Status,Iters,NormGrad,xFinal
,String,String,String,Int64,Float64,Array…
1,Sphere,Grad. Constante,MAX_ITER,10000,0.382709,"[0.135308, 0.135308]"
2,Sphere,Grad. Armijo,CONVERGIO,1,0.0,"[0.0, 0.0]"
3,Sphere,Newton,CONVERGIO,1,6.28037e-16,"[2.22045e-16, 2.22045e-16]"
4,Sphere,Grad. Conjugados,CONVERGIO,1,0.0,"[0.0, 0.0]"
5,Rosenbrock,Grad. Constante,MAX_ITER,10000,1.13845,"[0.322588, 0.100975]"
6,Rosenbrock,Grad. Armijo,CONVERGIO,960,9.91795e-7,"[0.999999, 0.999998]"
7,Rosenbrock,Newton,CONVERGIO,6,8.28571e-9,"[1.0, 1.0]"
8,Rosenbrock,Grad. Conjugados,CONVERGIO,217,9.89378e-7,"[0.999999, 0.999998]"
9,Rastrigin,Grad. Constante,CONVERGIO,368,9.63497e-7,"[0.994959, 0.994959]"
